# ABC Selective Attention - Structure-Sensitive Text Counting

Kaggle Bench runner for the counting task on the structure-sensitive text slice.


In [5]:
from dataclasses import dataclass

import kaggle_benchmarks as kbench
import pandas as pd

import abc_selective_attention_utils as utils

In [6]:
FAMILY = "selective attention"
ATTENTIONAL_BASIS = "structure sensitive"
MODALITY = "text"
TASK_TYPE = "counting"
TASK_NAME = f"{FAMILY} - {ATTENTIONAL_BASIS} {MODALITY} {TASK_TYPE}"

In [7]:
@dataclass
class CountAnswer:
    count: int

In [8]:
DATASET_ROOT = utils.DEFAULT_DATASET_ROOT
CSV_PATH = DATASET_ROOT / 'selective_attention/structure_sensitive/text/structure_sensitive_text_full.csv'

df = utils.load_csv(CSV_PATH)
print(CSV_PATH)
print('rows:', len(df))
print('columns:', sorted(df.columns.tolist()))


/kaggle/input/datasets/patrycjawegrzynowicz/abc-factorized-attention-benchmark/selective_attention/structure_sensitive/text/structure_sensitive_text_full.csv
rows: 680
columns: ['attentional_basis', 'binding_distance', 'confound_count', 'confound_type', 'count_instruction', 'count_prompt', 'dimension', 'family', 'filter_instruction', 'filter_prompt', 'gold_count', 'gold_lines', 'line_length_noise', 'modality', 'num_records', 'seed', 'serialization_style', 'structure_depth', 'structure_type', 'target_count', 'target_definition', 'text_input', 'unrelated_count', 'variant']


In [9]:
base_cols = [
    'seed',
    'family',
    'attentional_basis',
    'modality',
    'dimension',
    'variant',
    'principle',
    'render_style',
    'anchor_item_id',
    'anchor_group_id',
    'num_groups',
    'min_items_per_group',
    'max_items_per_group',
    'target_in_anchor_group',
    'target_outside_anchor_group',
    'non_target_in_anchor_group',
]

optional_cols = [
    col
    for col in [
        'format_styles',
        'section_style',
        'chain_style',
        'top_level_section_count',
        'nested_section_depth_max',
        'nested_section_probability',
        'chain_length_min',
        'chain_length_max',
    ]
    if col in df.columns
]

failure_cols = [
    'seed',
    'dimension',
    'variant',
    'principle',
    'anchor_item_id',
    'gold_count',
    'predicted_count',
    'failure_type',
]

count_eval_df = df[base_cols + optional_cols + ['count_prompt', 'gold_count']].copy()

groupings = [
    col
    for col in [
        'dimension',
        'variant',
        'principle',
        'render_style',
        'target_in_anchor_group',
        'section_style',
        'top_level_section_count',
        'nested_section_depth_max',
        'chain_style',
        'chain_length_min',
        'chain_length_max',
    ]
    if col in count_eval_df.columns
]

print('count_eval_df columns:', count_eval_df.columns.tolist())
print('groupings:', groupings)


count_eval_df columns: ['seed', 'family', 'attentional_basis', 'modality', 'dimension', 'variant', 'structure_type', 'structure_depth', 'binding_distance', 'serialization_style', 'num_records', 'target_count', 'confound_count', 'confound_type', 'line_length_noise', 'count_prompt', 'gold_count']
groupings: [['family'], ['family', 'attentional_basis'], ['family', 'attentional_basis', 'modality'], ['dimension'], ['dimension', 'variant'], ['structure_type'], ['num_records'], ['structure_depth'], ['structure_type', 'structure_depth'], ['binding_distance'], ['serialization_style'], ['target_count'], ['dimension', 'target_count'], ['confound_count'], ['confound_type'], ['structure_type', 'confound_type'], ['line_length_noise']]


In [10]:
@kbench.task(store_task=False)
def single_structure_sensitive_text_count(
    llm,
    seed,
    family,
    attentional_basis,
    modality,
    dimension,
    variant,
    principle,
    render_style,
    anchor_item_id,
    anchor_group_id,
    num_groups,
    min_items_per_group,
    max_items_per_group,
    target_in_anchor_group,
    target_outside_anchor_group,
    non_target_in_anchor_group,
    count_prompt,
    gold_count,
    format_styles=None,
    section_style=None,
    chain_style=None,
    top_level_section_count=None,
    nested_section_depth_max=None,
    nested_section_probability=None,
    chain_length_min=None,
    chain_length_max=None,
) -> dict:
    gold = int(gold_count)

    pred = None
    error = None
    error_type = None

    try:
        response = llm.prompt(count_prompt, schema=CountAnswer)
        pred = int(response.count)
    except Exception as exc:
        error_type, error = utils.classify_prompt_error(exc)

    is_correct = pred == gold
    has_error = error_type is not None
    is_failure = not is_correct
    failure_type = 'ok' if is_correct else (error_type if has_error else 'wrong_answer')

    kbench.assertions.assert_true(
        is_correct,
        expectation=f'Expected {gold}, got {pred}' + (f' | error_type={error_type} | error={error}' if error else ''),
    )

    result = {
        'task': 'structure_sensitive_text_count_v2',
        'model': llm.name,
        'seed': int(seed),
        'family': family,
        'attentional_basis': attentional_basis,
        'modality': modality,
        'dimension': dimension,
        'variant': variant,
        'principle': principle,
        'render_style': render_style,
        'task_type': 'counting',
        'anchor_item_id': anchor_item_id,
        'anchor_group_id': anchor_group_id,
        'num_groups': int(num_groups),
        'min_items_per_group': int(min_items_per_group),
        'max_items_per_group': int(max_items_per_group),
        'target_in_anchor_group': int(target_in_anchor_group),
        'target_outside_anchor_group': int(target_outside_anchor_group),
        'non_target_in_anchor_group': int(non_target_in_anchor_group),
        'gold_count': gold,
        'predicted_count': pred,
        'is_correct': is_correct,
        'is_failure': is_failure,
        'has_error': has_error,
        'failure_type': failure_type,
    }

    for name, value in [
        ('format_styles', format_styles),
        ('section_style', section_style),
        ('chain_style', chain_style),
        ('top_level_section_count', top_level_section_count),
        ('nested_section_depth_max', nested_section_depth_max),
        ('nested_section_probability', nested_section_probability),
        ('chain_length_min', chain_length_min),
        ('chain_length_max', chain_length_max),
    ]:
        if value not in (None, ''):
            result[name] = value

    if error_type is not None:
        result['error_type'] = error_type
    if error is not None:
        result['error'] = error

    return result


In [13]:
@kbench.task(name="selective attention - structure sensitive text counting")
def structure_sensitive_text_count_dataset(llm, df) -> tuple[float, float]:
    with kbench.client.enable_cache():
        runs = single_structure_sensitive_text_count.evaluate(
            stop_condition=lambda runs: len(runs) == df.shape[0],
            max_attempts=1,
            retry_delay=15,
            llm=[llm],
            evaluation_data=df,
            n_jobs=8,
            timeout=120,
            remove_run_files=False,
        )

    result = utils.summarize_and_log_runs(TASK_NAME, llm.name, runs, groupings, failure_cols)
    return result

In [14]:
run = structure_sensitive_text_count_dataset.run(kbench.llm, count_eval_df)
print(run)


[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.


=== selective attention - structure sensitive text counting : info ===
task: selective attention - structure sensitive text counting
model: google/gemini-2.5-flash
timestamp: 20260413_234709
run id: google_gemini_2.5_flash_20260413_234709
errors: 0 (rate 0.0)
overall: 1/1 = 100.00% (std 0.00%)
=== selective attention - structure sensitive text counting: by family ===
             family  passed  total  accuracy
selective_attention       1      1       1.0
=== selective attention - structure sensitive text counting: by family, attentional_basis ===
             family   attentional_basis  passed  total  accuracy
selective_attention structure_sensitive       1      1       1.0
=== selective attention - structure sensitive text counting: by family, attentional_basis, modality ===
             family   attentional_basis modality  passed  total  accuracy
selective_attention structure_sensitive     text       1      1       1.0
=== selective attention - structure sensitive text counting: by 

In [ ]:
%choose structure_sensitive_text_count_dataset
